# Advanced Multi-Objective Evaluation and Trust-Aware Ranking Framework for Personalized News Recommendation
## Academic Research & Machine Learning Training Pipeline

This notebook demonstrates the end-to-end data science and machine learning workflow for:
1. Loading authentic real-world news articles from the Kaggle ISOT Fake & Real News Dataset.
2. Rigorous text sanitization, including wire agency prefix stripping to prevent shortcut memorization.
3. Linguistic and rhetorical feature extraction (capitalization ratio, exclamation density, clickbait triggers).
4. Training and empirical evaluation of multiple classification models:
   - Logistic Regression
   - Passive Aggressive Classifier (Calibrated)
   - Multinomial Naive Bayes
   - Random Forest Classifier
5. Calculation of multi-factor Credibility Scores and Shannon Entropy Uncertainty.

In [ ]:
import os
import re
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
print('All libraries imported successfully!')

### 1. Data Ingestion & Sanitization
Loading raw `Fake.csv` and `True.csv` from the Kaggle ISOT dataset.

In [ ]:
fake_df = pd.read_csv('../dataset/Fake.csv')
true_df = pd.read_csv('../dataset/True.csv')
print(f'Raw Fake Articles: {len(fake_df):,}')
print(f'Raw True Articles: {len(true_df):,}')

fake_df['label'] = 'FAKE'
fake_df['target'] = 1
true_df['label'] = 'REAL'
true_df['target'] = 0

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.dropna(subset=['title', 'text']).drop_duplicates(subset=['title'])
print(f'Deduplicated Corpus: {len(df):,} unique articles')

### 2. Wire Prefix Stripping & Preprocessing
To ensure models learn real lexical cues rather than memorizing `(Reuters)` prefixes.

In [ ]:
WIRE_PATTERN = re.compile(r'^(?:[A-Z\s\/]{2,20})?\s*\((?:Reuters|AP|AFP|Associated Press|Bloomberg)\)\s*[-–—]\s*', re.I)

def clean_article(text):
    if not isinstance(text, str): return ''
    text = WIRE_PATTERN.sub('', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<.*?>+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_clean'] = df['text'].apply(clean_article)
df['content'] = df['title'] + ' ' + df['text_clean']
print('Sample sanitized text:', df['content'].iloc[0][:150])

### 3. Stratified Train / Validation / Test Splitting

In [ ]:
X = df['content']
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set size: {len(X_train):,}')
print(f'Testing set size : {len(X_test):,}')

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english', sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(f'TF-IDF Feature Matrix: {X_train_vec.shape}')

### 4. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=1.5, max_iter=1000, random_state=42),
    'Passive Aggressive': CalibratedClassifierCV(PassiveAggressiveClassifier(C=0.5, max_iter=1000, random_state=42), cv=3),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=30, n_jobs=-1, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results[name] = {'Accuracy': acc, 'F1-Score': f1}
    print(f'{name:25s} | Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%')

results_df = pd.DataFrame(results).T
results_df.plot(kind='bar', figsize=(9, 4), ylim=(0.9, 1.0))
plt.title('Classifier Performance Comparison on Test Set')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 5. Multi-Factor Credibility Scoring Function
Demonstration of calculating unified credibility incorporating classifier probability, source reputation, and sensationalism.

In [ ]:
def demo_credibility(prob_real, source_trust, sensationalism):
    score = 0.55 * prob_real + 0.30 * source_trust - 0.15 * sensationalism
    return max(0.01, min(0.99, score))

print('Credibility for Verified Reuters article (prob=0.99, source=0.95, sens=0.10):', 
      round(demo_credibility(0.99, 0.95, 0.10), 3))
print('Credibility for Viral Blog article (prob=0.15, source=0.25, sens=0.85):', 
      round(demo_credibility(0.15, 0.25, 0.85), 3))